# Import Data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/IndividualProject/cxr"
HDF5_PATH = f"{BASE_DIR}/images.hdf5"

In [ ]:
# import the required libraries
import numpy as np
import pandas as pd
import os
import ast

In [ ]:
processed_df = pd.read_csv(BASE_DIR + "/processed_data_v3.csv")
processed_df.head()

,filepath,split,tasks/disease labels,tasks/patient sex,original_filepath,original_split,patient_id,bounding_box,disease labels,finding_labels,follow-up_nb,original_image_size,original_pixel_spacing,patient sex,patient_age,view_position,tasks/atelectasis_label,tasks/pneumonia_label
0,images/000000.tiff,train,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,images/00000001_000.png,train,1,NaN,Cardiomegaly,Cardiomegaly,0,"(2682,2749)","(0.143,0.143)",M,57,PA,0,0
1,images/000001.tiff,train,"[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]",1,images/00000001_001.png,train,1,NaN,Cardiomegaly|Emphysema,Cardiomegaly|Emphysema,1,"(2894,2729)","(0.143,0.143)",M,58,PA,0,0
2,images/000002.tiff,train,"[0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,images/00000001_002.png,train,1,NaN,Cardiomegaly|Effusion,Cardiomegaly|Effusion,2,"(2500,2048)","(0.168,0.168)",M,58,PA,0,0
3,images/000003.tiff,train,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]",1,images/00000002_000.png,train,2,NaN,NaN,No Finding,0,"(2500,2048)","(0.171,0.171)",M,80,PA,0,0
4,images/000004.tiff,test,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]",0,images/00000003_001.png,test,3,NaN,Hernia,Hernia,0,"(2500,2048)","(0.168,0.168)",F,74,PA,0,0


In [ ]:
# read the images in the train set
with open(BASE_DIR + "/splits/train.txt", "r") as f:
    train_list = [image_filepath.strip() for image_filepath in f]

# retrieve the entries in the train split and convert them into a DataFrame
train_df = processed_df[processed_df["filepath"].isin(train_list)]

# repeat the same process for validation set
with open(BASE_DIR + "/splits/val.txt", "r") as f:
    val_list = [image_filepath.strip() for image_filepath in f]

val_df = processed_df[processed_df["filepath"].isin(val_list)]

# repeat the same for test set
with open(BASE_DIR + "/splits/test.txt", "r") as f:
    test_list = [image_filepath.strip() for image_filepath in f]

test_df = processed_df[processed_df["filepath"].isin(test_list)]


# Adversarial Unlearning for Age


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
import torch.optim as optim
import h5py
from PIL import Image
from tqdm import tqdm
from torch.autograd import Function
import torch.nn.functional as F

from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix, roc_curve

In [ ]:
class ChestXRayDataset(Dataset):

    def __init__(self, df, labels_col, hdf5_path, transform=None):
        self.df = df
        self.labels_col = labels_col
        self.hdf5_path = hdf5_path
        self.transform = transform

        self.h5 = h5py.File(self.hdf5_path, "r")
        self.images = self.h5["images"]

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        original_index = self.df.index[idx]
        img = self.images[original_index]
        img = torch.tensor(img, dtype=torch.float32)

        if img.max() > 1:
            img = img / 255.0

        img = img.unsqueeze(0).repeat(3,1,1)

        if self.transform:
            img = self.transform(img)

        y = float(self.df.iloc[idx][self.labels_col])
        label = torch.tensor([y], dtype=torch.float32)

        sex = float(self.df.iloc[idx]["tasks/patient sex"])
        sex_label = torch.tensor([sex], dtype=torch.float32)

        raw_age = self.df.iloc[idx]['patient_age']

        if raw_age >= 50:
          age_binary = 1
        else:
          age_binary = 0

        age = torch.tensor(float(age_binary), dtype=torch.float32)
        age = age.unsqueeze(0)

        return img, label, sex_label, age

In [ ]:
# data augmentation process
from torchvision import transforms

# transform function for train set
train_transform = transforms.Compose([
    # perform horizontal flip, rotation and normalize
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])

]
)

# trasform function for validation set
valid_transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# transform function for test set
test_transform = transforms.Compose([
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [ ]:
train_dataset = ChestXRayDataset(
    df = train_df,
    labels_col = 'tasks/atelectasis_label',
    hdf5_path=HDF5_PATH,
    transform = train_transform
)

valid_dataset = ChestXRayDataset(
    df = val_df,
    labels_col = 'tasks/atelectasis_label',
    hdf5_path=HDF5_PATH,
    transform = valid_transform
)

test_dataset = ChestXRayDataset(
    df = test_df,
    labels_col = 'tasks/atelectasis_label',
    hdf5_path=HDF5_PATH,
    transform = test_transform
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size = 64,
    shuffle = True,
    num_workers = 0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size = 64,
    shuffle = False,
    num_workers = 0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size = 64,
    shuffle = False,
    num_workers = 0,
    pin_memory=True
)

In [ ]:
# define Gradient Reversal Layer for adversarial training
# reference: https://discuss.pytorch.org/t/solved-reverse-gradients-in-backward-pass/3589/26
# reference: https://stackoverflow.com/questions/66814765/gradreverselayer-notimplementederror-you-must-implement-the-backward-function
# reference: https://github.com/fungtion/DANN/blob/master/models/functions.py
class ReverseLayerF(Function):

  @staticmethod
  def forward(ctx, x, lambd):
    ctx.lambd = lambd
    return x.view_as(x)

  # makes the gradient negative to make age prediction works
  @staticmethod
  def backward(ctx, grad_output):
    output = grad_output.neg() * ctx.lambd
    return output, None

# # the gradient reversal layer
# def grl(x, lambd):
#   return ReverseLayerF.apply(x, lambd)

In [ ]:
class AdversarialDenseNet121(nn.Module):
  def __init__(self):
    super(AdversarialDenseNet121, self).__init__()
    # pretrained model - same as baseline
    backbone = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    self.features = backbone.features
    feat_dim = backbone.classifier.in_features

    # predict atelectasis
    # disease classifier C (1 = has atelectasis)
    self.task_head = nn.Sequential(nn.Dropout(0.4), nn.Linear(feat_dim, 1))


    # predict sex
    # bias discriminator D
    self.sex_head = nn.Sequential(nn.Dropout(0.3),nn.Linear(feat_dim, 512),nn.ReLU(inplace=True),nn.Linear(512, 1))
    # self.sex_head = nn.Sequential(nn.Dropout(0.4), nn.Linear(feat_dim, 1))

    self.age_head = nn.Sequential(nn.Dropout(0.3), nn.Linear(feat_dim, 512), nn.ReLU(inplace=True), nn.Linear(512, 1))

    # passes input image through DenseNet convolutional layers
  def extract(self, x):
    h = self.features(x)
    h = F.relu(h, inplace=True)
    h = F.adaptive_avg_pool2d(h, (1,1))
    h = h.view(h.size(0), -1)
    return h

  def forward(self, input_data, lambd):

    # extract feature vector
    feature = self.extract(input_data)
    reverse_feature = ReverseLayerF.apply(feature, lambd)

    # send the extracted h to effusion classifier (yes/no)
    task_output = self.task_head(feature)

    # send the extracted h to age classifier (male/female) - includes GRL layer
    sex_output = self.sex_head(reverse_feature)
    age_output =self.age_head(reverse_feature)
    return task_output, sex_output, age_output

In [ ]:
# reference: https://github.com/Yangyangii/DANN-pytorch/blob/master/DANN.ipynb
def lambda_schedule(epoch, max_epoch, max_lambd):

  p = epoch/max_epoch
  return float(max_lambd * (2 / (1 + np.exp(-10 * p)) - 1))

In [ ]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

# model = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
model = AdversarialDenseNet121().to(DEVICE)

# pos_weight
pos = float((train_df["tasks/atelectasis_label"] == 1).sum())
neg = float((train_df["tasks/atelectasis_label"] == 0).sum())
pos_weight = torch.tensor([neg/ max(pos,1.0)], device =DEVICE)

# using binary cross entropy with logits loss
# criterion = nn.BCEWithLogitsLoss()
task_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
# since it is not really imbalance so do not need pos_weight
sex_criterion = nn.BCEWithLogitsLoss()
age_criterion = nn.BCEWithLogitsLoss()

# num_features = model.classifier.in_features
# model.classifier = nn.Sequential(nn.Dropout(0.4), nn.Linear(num_features, 1))
# model = model.to(DEVICE)

learning_rate = 5e-4
optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

Device: cuda
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 198MB/s]


In [ ]:
def train_one_epoch(model, loader, task_criterion, sex_criterion, age_criterion, optimizer, device, lambd):
  # use_amp = device.type == "cuda"

  model.train()
  training_loss, sex_loss, age_loss = 0.0, 0.0, 0.0
  loop = tqdm(loader, desc="Training", leave=False)

  for images, labels, sex, age in loop:
    images = images.to(device, non_blocking=True)
    labels = labels.to(device, non_blocking=True)
    sex = sex.to(device, non_blocking=True)
    age = age.to(device, non_blocking=True)
    age = age.float().view(-1, 1)

    # outputs = model(images)
      # compute the loss
    # loss = criterion(outputs, labels)
    optimizer.zero_grad(set_to_none=True)
    task_logit, sex_logit, age_logit = model(images, lambd=lambd)
    loss_task = task_criterion(task_logit, labels)
    loss_sex = sex_criterion(sex_logit, sex)
    loss_age = age_criterion(age_logit, age)

    loss = loss_task + loss_sex+ loss_age
    loss.backward()
    optimizer.step()

    training_loss += loss_task.item()
    sex_loss += loss_sex.item()
    age_loss += loss_age.item()

  avg_training_loss = training_loss/len(loader)
  avg_sex_loss = sex_loss/len(loader)
  avg_age_loss = age_loss/len(loader)

  return avg_training_loss, avg_sex_loss,avg_age_loss

In [ ]:
def validate(model, loader, task_criterion, sex_criterion, age_criterion, device):
    model.eval()
    validation_loss, sex_loss, age_loss = 0.0, 0.0, 0.0
    validation_preds, validation_labels = [], []
    sex_preds, sex_labels = [], []
    age_preds, age_labels = [], []

    # disable gradient calculation
    with torch.no_grad():
      loop = tqdm(loader, desc="Validation", leave=False)

      for images, labels, sex, age in loop:
          images = images.to(device, non_blocking=True)
          labels = labels.to(device, non_blocking=True)
          sex = sex.to(device, non_blocking=True)
          age = age.to(device, non_blocking=True)
          age = age.float().view(-1, 1)

          task_logit, sex_logit, age_logit = model(images, lambd=0.0)
          validation_loss += task_criterion(task_logit, labels).item()
          sex_loss += sex_criterion(sex_logit, sex).item()
          age_loss += age_criterion(age_logit, age).item()

          validation_preds.append(torch.sigmoid(task_logit).cpu())
          validation_labels.append(labels.cpu())
          sex_preds.append(torch.sigmoid(sex_logit).cpu())
          sex_labels.append(sex.cpu())
          age_preds.append(torch.sigmoid(age_logit).cpu())
          age_labels.append(age.cpu())

    validation_preds = torch.cat(validation_preds).numpy().reshape(-1)
    validation_labels = torch.cat(validation_labels).numpy().reshape(-1)
    sex_preds = torch.cat(sex_preds).numpy().reshape(-1)
    sex_labels = torch.cat(sex_labels).numpy().reshape(-1)
    age_preds = torch.cat(age_preds).numpy().reshape(-1)
    age_labels = torch.cat(age_labels).numpy().reshape(-1)

    # compute the auc score directly
    valid_auc = roc_auc_score(validation_labels, validation_preds) if len(np.unique(validation_labels)) > 1 else np.nan
    sex_auc = roc_auc_score(sex_labels, sex_preds) if len(np.unique(sex_labels)) > 1 else np.nan
    age_auc = roc_auc_score(age_labels, age_preds) if len(np.unique(age_labels)) > 1 else np.nan

    return validation_loss / len(loader), sex_loss/len(loader), age_loss/len(loader), validation_preds, validation_labels, sex_preds, sex_labels, age_preds, age_labels, valid_auc, sex_auc, age_auc

In [ ]:
def youden_threshold(y_true, y_prob):
  fpr_val, tpr_val, thresholds = roc_curve(y_true, y_prob)

  j_scores = tpr_val - fpr_val
  youden_index = np.argmax(j_scores)
  optimal_threshold = thresholds[youden_index]

  return optimal_threshold

In [ ]:
def save_checkpoint(epoch, model, optimizer, learning_rate, filename):
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'learning_rate': learning_rate
    }
    torch.save(checkpoint, filename)

In [ ]:
from datetime import datetime

EPOCHS = 30
best_val = -1.0
patience = 5
min_delta = 0.001
epochs_no_improve = 0
training_loss, validation_loss, validation_auc = [], [], []
training_sex_loss, validation_sex_loss, validation_sex_auc = [], [], []
training_age_loss, validation_age_loss, validation_age_auc = [], [], []
lambd_values = []

# save for model checkpoint
for epoch in range(1, EPOCHS+1):
  lambd = lambda_schedule(epoch-1, EPOCHS, max_lambd=1.0)
  #  scaler
  train_loss, train_sex_loss, train_age_loss = train_one_epoch(model, train_loader, task_criterion, sex_criterion, age_criterion, optimizer, DEVICE, lambd)
  (val_loss, val_sex_loss, val_age_loss, val_preds, val_labels, sex_preds, sex_labels, age_preds, age_labels, val_auc, sex_auc, age_auc) = validate(model, valid_loader, task_criterion, sex_criterion, age_criterion, DEVICE)

  training_loss.append(train_loss)
  validation_loss.append(val_loss)
  validation_auc.append(val_auc)
  training_sex_loss.append(train_sex_loss)
  validation_sex_loss.append(val_sex_loss)
  validation_sex_auc.append(sex_auc)
  training_age_loss.append(train_age_loss)
  validation_age_loss.append(val_age_loss)
  validation_age_auc.append(age_auc)
  lambd_values.append(lambd)

  scheduler.step(val_auc if not np.isnan(val_auc) else -1.0)

  print(f"Epoch {epoch}/{EPOCHS} | λ={lambd:.3f} \n"
          f"Training Task Loss={train_loss:.4f}| Training Sex Loss={train_sex_loss:.4f} | Training age Loss={train_age_loss:.4f} \n"
          f"Validation Task Loss={val_loss:.4f}| Validation Sex Loss={train_sex_loss:.4f} | Validation age Loss={val_age_loss:.4f} \n"
          f"Val task AUC={val_auc:.4f} | Val sex AUC={sex_auc:.4f}| Val age AUC={age_auc:.4f}")

  # implement early stopping
  if not np.isnan(val_auc) and val_auc > best_val + min_delta:
      best_val = val_auc
      best_epoch = epoch
      epochs_no_improve = 0
      timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
      torch.save(model.state_dict(), f"/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_sex_age_lambda1.pth")
  else:
      epochs_no_improve += 1
      if epochs_no_improve >= patience:
          print(f"Early stopping: no val AUC improvement for {patience} epochs.")
          break

  save_checkpoint(epoch, model, optimizer, learning_rate, f"checkpoint_epoch_{epoch}.pth")

In [ ]:
# save the best result
import json
import os
from datetime import datetime

SAVE_DIR = "/content/drive/MyDrive/IndividualProject/results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Create training history dictionary
training_history = {
    'model_name': 'adversarial_densenet121',
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'hyperparameters': {
        'epochs': EPOCHS,
        'batch_size': 64,
        'learning_rate': learning_rate,
        'weight_decay': 1e-4,
        'optimizer': 'Adam',
        'scheduler': 'ReduceLROnPlateau',
        'patience': patience,
        'pos_weight': float(pos_weight.cpu().item()) if torch.is_tensor(pos_weight) else float(pos_weight)
    },
    'dataset_sizes': {
        'train': len(train_df),
        'validation': len(val_df),
        'test': len(test_df)
    },
    'training_curves': {
        'epochs': list(range(1, len(training_loss) + 1)),
        'train_loss': [float(x) for x in training_loss],
        'val_loss': [float(x) for x in validation_loss],
        'val_auc': [float(x) for x in validation_auc],
        'train_sex_loss': [float(x) for x in training_sex_loss],
        'val_sex_loss': [float(x) for x in validation_sex_loss],
        'val_sex_auc': [float(x) for x in validation_sex_auc],
        'train_age_loss': [float(x) for x in training_age_loss],
        'val_age_loss': [float(x) for x in validation_age_loss],
        'val_age_auc':[float(x) for x in validation_age_auc],
        'lambd': lambd_values
    },
    'best_model': {
        'epoch': best_epoch,
        'val_auc': float(max(validation_auc)),
        'val_loss': float(validation_loss[np.argmax(validation_auc)]),
        'val_sex_loss': float(validation_sex_loss[np.argmax(validation_auc)]),
        'val_sex_auc': float(validation_sex_auc[np.argmax(validation_auc)]),
        'val_age_auc': float(validation_age_auc[np.argmax(validation_auc)]),
        'val_age_loss': float(validation_age_loss[np.argmax(validation_auc)])
    },

    'early_stopping': {
        'triggered': epochs_no_improve >= patience,
        'stopped_at_epoch': len(training_loss)
    }
}

# Save as JSON
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
history_file = f"{SAVE_DIR}/adversarial_training_history_atelectasis_sex_age_lambda1.json"
with open(history_file, 'w') as f:
    json.dump(training_history, f, indent=4)

print(f"Training history saved to {history_file}")

# Also save a summary
print("\n" + "="*60)
print("TRAINING SUMMARY")
print("="*60)
print(f"Total epochs trained: {len(training_loss)}")
print(f"Best epoch: {training_history['best_model']['epoch']}")
print(f"Best validation AUC: {training_history['best_model']['val_auc']:.4f}")
print(f"Early stopping: {'Yes' if training_history['early_stopping']['triggered'] else 'No'}")
print("="*60)

In [ ]:
# plot training vs validation loss for disease classifier
import matplotlib.pyplot as plt

total_epochs = len(training_loss)

plt.figure(figsize=(10, 6))
plt.plot(range(1, total_epochs+1), training_loss, label="Training Loss")
plt.plot(range(1, total_epochs+1), validation_loss, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss for Atelectasis")
plt.legend()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# plt.savefig(f"/content/drive/MyDrive/IndividualProject/results/figures/adversarial_training_vs_validation_loss_task_age{timestamp}.png")
plt.show()

In [ ]:
# plot training vs validation loss for age classifier
import matplotlib.pyplot as plt

total_epochs = len(training_loss)

plt.figure(figsize=(10, 6))
plt.plot(range(1, total_epochs+1), training_age_loss, label="Training Loss")
plt.plot(range(1, total_epochs+1), validation_age_loss, label="Validation Loss")

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss for age")
plt.legend()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# plt.savefig(f"/content/drive/MyDrive/IndividualProject/results/figures/adversarial_training_vs_validation_loss_age_{timestamp}.png")
plt.show()

In [ ]:
# plot disease validation auc
plt.figure()
plt.plot(range(1, total_epochs+1), validation_auc)

plt.xlabel("Epoch")
plt.ylabel("Validation AUC")
plt.title("Validation AUC vs Epoch")
plt.legend()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# plt.savefig(f"/content/drive/MyDrive/IndividualProject/results/figures/adverarial_validation_auc_vs_epoch_task{timestamp}.png")
plt.show()

In [ ]:
# plot age validation auc
plt.figure()
plt.plot(range(1, total_epochs+1), validation_age_auc)

plt.xlabel("Epoch")
plt.ylabel("Validation Age AUC")
plt.title("Validation age AUC vs Epoch")
plt.legend()

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# plt.savefig(f"/content/drive/MyDrive/IndividualProject/results/figures/adverarial_validation_age_auc_vs_epoch_{timestamp}.png")
plt.show()

In [ ]:
from sklearn.metrics import recall_score

def gender_disparity(y_prob, y_true, df, threshold):

  # initialise a dictionaries to store the result
  disparities = {}

  # turn the probabilities into binary classification
  y_pred = (y_prob >= threshold).astype(int)


  # male_group = (df['patient sex'] == 'M').to_numpy()
  # female_group = (df['patient sex'] == 'F').to_numpy()

  male_group = (df['tasks/patient sex'] == 1).to_numpy()
  female_group = (df['tasks/patient sex'] == 0).to_numpy()

  # group them baseed on male/female
  male_preds = y_pred[male_group]
  male_true = y_true[male_group]

  female_preds = y_pred[female_group]
  female_true = y_true[female_group]

  # skip if either male/female group has no positive result
  if male_true.sum() == 0 or female_true.sum() == 0:
    return None

  # compute the tpr for male and female group
  male_tpr = recall_score(male_true, male_preds)
  female_tpr = recall_score(female_true, female_preds)

  # compute the difference
  # this will shows a negative result if female group is disadvantaged
  disparity = female_tpr - male_tpr

  # store the result in dictionary
  disparities["atelectasis"] = {
    'tpr_male': round(float(male_tpr), 4),
    'tpr_female': round(float(female_tpr), 4),
    'disparity': round(float(disparity), 4),
    'male_samples': int(male_group.sum()),
    'female_samples': int(female_group.sum()),
    'total_samples': int(len(y_pred))
  }

  return disparities

In [ ]:
def auc_by_sex(y_prob, y_true, df):

    results = {}

    male_group = (df['tasks/patient sex'] == 1).to_numpy()
    female_group = (df['tasks/patient sex'] == 0).to_numpy()

    # Male AUC
    if len(np.unique(y_true[male_group])) > 1:
        auc_male = roc_auc_score(y_true[male_group], y_prob[male_group])
    else:
        auc_male = np.nan

    # Female AUC
    if len(np.unique(y_true[female_group])) > 1:
        auc_female = roc_auc_score(y_true[female_group], y_prob[female_group])
    else:
        auc_female = np.nan

    results["atelectasis"] = {
    "auc_male": round(float(auc_male), 4),
    "auc_female": round(float(auc_female), 4),
    "auc_gap": round(float(abs(auc_male - auc_female)), 4)
  }


    return results


In [ ]:
from sklearn.metrics import recall_score
def age_disparity(y_prob, y_true, df, threshold):

  disparities = {}

  y_pred = (y_prob >= threshold).astype(int)

  young = (df['patient_age'] < 50).to_numpy()
  old = (df['patient_age'] >= 50).to_numpy()

  young_true = y_true[young]
  young_pred = y_pred[young]

  old_true = y_true[old]
  old_pred = y_pred[old]

  young_tpr = recall_score(young_true, young_pred)
  old_tpr = recall_score(old_true, old_pred)

  tpr_disparity = old_tpr - young_tpr

  disparities["atelectasis"] = {
    'tpr_young': round(float(young_tpr), 4),
    'tpr_old': round(float(old_tpr), 4),
    'tpr_disparity': round(float(tpr_disparity), 4),
    'young_samples': int(young.sum()),
    'old_samples': int(old.sum()),
    'total_samples': int(len(y_pred))
  }

  return disparities

In [ ]:
def auc_by_age(y_prob, y_true, df):

    results = {}

    young = (df['patient_age'] < 50).to_numpy()
    old = (df['patient_age'] >= 50).to_numpy()

    young_true = y_true[young]
    young_prob = y_prob[young]

    old_true = y_true[old]
    old_prob = y_prob[old]

    auc_young = roc_auc_score(young_true, young_prob)
    auc_old = roc_auc_score(old_true, old_prob)

    auc_gap = abs(auc_old - auc_young)

    results["atelectasis"] = {
    "auc_young": round(float(auc_young), 4),
    "auc_old": round(float(auc_old), 4),
    "auc_gap": round(float(auc_gap), 4)
  }


    return results

In [ ]:
from sklearn.metrics import recall_score
import numpy as np

def intersectional_disparity(y_prob, y_true, df, threshold):

    disparities = {}

    # binarise predictions
    y_pred = (y_prob >= threshold).astype(int)

    # define groups
    male = (df['tasks/patient sex'] == 1).to_numpy()
    female = (df['tasks/patient sex'] == 0).to_numpy()

    young = (df['patient_age'] < 50).to_numpy()
    old = (df['patient_age'] >= 50).to_numpy()

    # intersectional masks
    male_young = male & young
    male_old = male & old
    female_young = female & young
    female_old = female & old

    groups = {
        "male_<50": male_young,
        "male_>=50": male_old,
        "female_<50": female_young,
        "female_>=50": female_old
    }

    tprs = {}

    # compute TPR per group
    for name, mask in groups.items():
        group_true = y_true[mask]
        group_pred = y_pred[mask]

        if group_true.sum() == 0:
            tpr = np.nan  # avoid divide by zero
        else:
            tpr = recall_score(group_true, group_pred)

        tprs[name] = tpr

    # compute median TPR (ignore NaN)
    valid_tprs = [v for v in tprs.values() if not np.isnan(v)]
    median_tpr = np.median(valid_tprs)

    # compute disparities (median-based)
    gaps = {}
    for k, v in tprs.items():
        if np.isnan(v):
            gaps[k] = np.nan
        else:
            gaps[k] = v - median_tpr

    # overall intersectional gap (max - min)
    if len(valid_tprs) > 0:
        max_min_gap = max(valid_tprs) - min(valid_tprs)
    else:
        max_min_gap = np.nan

    disparities["atelectasis"] = {
        "tpr_per_group": {k: round(float(v), 4) if not np.isnan(v) else None for k, v in tprs.items()},
        "median_tpr": round(float(median_tpr), 4),
        "gap_per_group": {k: round(float(v), 4) if not np.isnan(v) else None for k, v in gaps.items()},
        "max_min_tpr_gap": round(float(max_min_gap), 4),
        "group_sizes": {k: int(mask.sum()) for k, mask in groups.items()}
    }

    return disparities

In [ ]:
from sklearn.metrics import roc_auc_score
import numpy as np

def auc_by_intersection(y_prob, y_true, df):

    results = {}

    # define sex masks
    male = (df['tasks/patient sex'] == 1).to_numpy()
    female = (df['tasks/patient sex'] == 0).to_numpy()

    # define age masks
    young = (df['patient_age'] < 50).to_numpy()
    old = (df['patient_age'] >= 50).to_numpy()

    # intersectional masks
    groups = {
        "male_<50": male & young,
        "male_>=50": male & old,
        "female_<50": female & young,
        "female_>=50": female & old
    }

    aucs = {}

    for name, mask in groups.items():
        group_true = y_true[mask]
        group_prob = y_prob[mask]

        # AUC is only valid if both classes are present
        if len(np.unique(group_true)) < 2:
            auc = np.nan
        else:
            auc = roc_auc_score(group_true, group_prob)

        aucs[name] = auc

    # ignore NaNs
    valid_aucs = [v for v in aucs.values() if not np.isnan(v)]

    if len(valid_aucs) > 0:
        median_auc = np.median(valid_aucs)
        auc_range = max(valid_aucs) - min(valid_aucs)
    else:
        median_auc = np.nan
        auc_range = np.nan

    # disparity from median
    auc_gaps = {}
    for k, v in aucs.items():
        if np.isnan(v):
            auc_gaps[k] = np.nan
        else:
            auc_gaps[k] = v - median_auc

    results["atelectasis"] = {
        "auc_per_group": {
            k: round(float(v), 4) if not np.isnan(v) else None
            for k, v in aucs.items()
        },
        "median_auc": round(float(median_auc), 4) if not np.isnan(median_auc) else None,
        "auc_gap_per_group": {
            k: round(float(v), 4) if not np.isnan(v) else None
            for k, v in auc_gaps.items()
        },
        "max_min_auc_gap": round(float(auc_range), 4) if not np.isnan(auc_range) else None,
        "group_sizes": {
            k: int(mask.sum()) for k, mask in groups.items()
        }
    }

    return results

In [ ]:
# recreate the model
BEST_MODEL_PATH =  "/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_sex_age_lambda1.pth"

model = AdversarialDenseNet121().to(DEVICE)

best_model = torch.load(BEST_MODEL_PATH, map_location=DEVICE)

model.load_state_dict(best_model)
model.eval()

(val_loss, val_sex_loss, val_age_loss, val_preds, val_labels,
 sex_preds, sex_labels, age_preds, age_labels, val_auc, sex_auc, age_auc) = validate(model, valid_loader, task_criterion, sex_criterion, age_criterion, DEVICE)

best_threshold = youden_threshold(val_labels, val_preds)
print("Optimal threshold:", best_threshold)

(test_loss, test_sex_loss, test_age_loss, test_preds, test_labels,
 test_sex_preds, test_sex_labels, test_age_preds, test_age_labels, test_auc, test_sex_auc, test_age_auc) = validate(
    model, test_loader, task_criterion, sex_criterion, age_criterion, DEVICE)

# print out the result
print(f"Test Loss: {test_loss:.4f} | Test AUC: {test_auc:.4f} | Sex AUC: {test_sex_auc:.4f}| Age AUC: {test_age_auc:.4f}")

test_sex_gap = gender_disparity(test_preds, test_labels, test_df, threshold=best_threshold)
print("Gender gap:", test_sex_gap)

test_auc_by_sex = auc_by_sex(test_preds, test_labels, test_df)
print("AUC by sex:", test_auc_by_sex)

# compute tpr
test_age_gap = age_disparity(test_preds, test_labels, test_df, threshold=best_threshold)
print("Age gap:", test_age_gap)

test_auc_by_age = auc_by_age(test_preds, test_labels, test_df)
print("AUC by age:", test_auc_by_age)


test_intersectional_tpr = intersectional_disparity(test_preds, test_labels, test_df, threshold = best_threshold)
print("Intersectional TPR disparity:", test_intersectional_tpr)

test_intersection_auc = auc_by_intersection(
    test_preds, test_labels, test_df)
print("Intersectional AUC:", test_intersection_auc)



Optimal threshold: 0.5030759


Test Loss: 1.2299 | Test AUC: 0.7501 | Sex AUC: 0.5501| Age AUC: 0.5775
Gender gap: {'atelectasis': {'tpr_male': 0.8194, 'tpr_female': 0.8034, 'disparity': -0.016, 'male_samples': 14882, 'female_samples': 10714, 'total_samples': 25596}}
AUC by sex: {'atelectasis': {'auc_male': 0.7616, 'auc_female': 0.7332, 'auc_gap': 0.0284}}
Age gap: {'atelectasis': {'tpr_young': 0.7869, 'tpr_old': 0.8332, 'tpr_disparity': 0.0463, 'young_samples': 13167, 'old_samples': 12429, 'total_samples': 25596}}
AUC by age: {'atelectasis': {'auc_young': 0.7579, 'auc_old': 0.7384, 'auc_gap': 0.0194}}
Intersectional TPR disparity: {'atelectasis': {'tpr_per_group': {'male_<50': 0.7871, 'male_>=50': 0.843, 'female_<50': 0.7865, 'female_>=50': 0.8182}, 'median_tpr': 0.8027, 'gap_per_group': {'male_<50': -0.0155, 'male_>=50': 0.0403, 'female_<50': -0.0162, 'female_>=50': 0.0155}, 'max_min_tpr_gap': 0.0565, 'group_sizes': {'male_<50': 7634, 'male_>=50': 7248, 'female_<50': 5533, 'female_>=50': 5181}}}
Intersectional AUC

In [ ]:
import json
import pickle
import pandas as pd
from datetime import datetime

# ===== Save Results to Drive =====
SAVE_DIR = "/content/drive/MyDrive/IndividualProject/results"
os.makedirs(SAVE_DIR, exist_ok=True)

# Create results dictionary
baseline_results = {
    'model_name': 'adversarial_densenet121',
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'dataset_size': {
        'train': len(train_df),
        'val': len(val_df),
        'test': len(test_df)
    },
    'test_metrics': {
        'test_loss': float(test_loss),
        'test_auc': float(test_auc),
        'test_sex_loss': float(test_sex_loss),
        'test_sex_auc': float(test_sex_auc),
        'test_age_loss': float(test_age_loss),
        'test_age_auc': float(test_age_auc),
        'threshold': float(best_threshold)
    },
    'fairness_metrics': {
        'tpr_male': test_sex_gap['atelectasis']['tpr_male'],
        'tpr_female': test_sex_gap['atelectasis']['tpr_female'],
        'tpr_sex_disparity': test_sex_gap['atelectasis']['disparity'],
        'tpr_young': test_age_gap['atelectasis']['tpr_young'],
        'tpr_old': test_age_gap['atelectasis']['tpr_old'],
        'tpr_disparity': test_age_gap['atelectasis']['tpr_disparity'],
        'auc_male': test_auc_by_sex['atelectasis']['auc_male'],
        'auc_female': test_auc_by_sex['atelectasis']['auc_female'],
        'auc_gap': test_auc_by_sex['atelectasis']['auc_gap'],
        'auc_young': test_auc_by_age['atelectasis']['auc_young'],
        'auc_old': test_auc_by_age['atelectasis']['auc_old'],
        'auc_gap': test_auc_by_age['atelectasis']['auc_gap']
    },
    'sample_distribution': {
        'male_samples:': test_sex_gap['atelectasis']['male_samples'],
        'female_samples': test_sex_gap['atelectasis']['female_samples'],
        'total_samples': test_sex_gap['atelectasis']['total_samples'],
        'young_samples': test_age_gap['atelectasis']['young_samples'],
        'old_samples': test_age_gap['atelectasis']['old_samples'],
        'total_samples': test_age_gap['atelectasis']['total_samples']
    }
}

# Save as JSON (human-readable)
with open(f"{SAVE_DIR}/adversarial_results_actelatasis_youden_sex_age_lambda1.json", 'w') as f:
    json.dump(baseline_results, f, indent=4)
print(f"Results saved to {SAVE_DIR}/baseline_adversarial_results_actelatasis_youden_sex_age_lambda1.json")


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

young_mask = (test_df['patient_age'] < 50).to_numpy()
old_mask = (test_df['patient_age'] >= 50).to_numpy()

fpr_young, tpr_young, _ = roc_curve(test_labels[young_mask],
                                  test_preds[young_mask])
young_auc = roc_auc_score(test_labels[young_mask],
                         test_preds[young_mask])

fpr_old, tpr_old, _ = roc_curve(test_labels[old_mask],
                                  test_preds[old_mask])
old_auc = roc_auc_score(test_labels[old_mask],
                         test_preds[old_mask])


plt.figure(figsize=(6,6))

plt.plot(fpr_young, tpr_young,
         label=f'Young (AUC = {young_auc:.3f})')

plt.plot(fpr_old, tpr_old,
         label=f'Old (AUC = {old_auc:.3f})')

plt.plot([0,1], [0,1], linestyle='--', color='gray')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Adversarial ROC by Age')
plt.legend()
plt.tight_layout()

# timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
# plt.savefig(f"/content/drive/MyDrive/IndividualProject/results/figures/baseline_roc_by_sex_{timestamp}.png")
plt.show()

# Sex Probing


In [ ]:

# define extract features functions
def extract_features(model, loader, device):

  model.eval()
  all_features = []
  all_labels = []
  all_sex = []

  with torch.no_grad():
    for imgs, labels, sex, _ in loader:
      imgs = imgs.to(device)

      features = model.extract(imgs)

      all_features.append(features.cpu().numpy())
      all_labels.append(labels.numpy())
      all_sex.append(sex.numpy())


  all_features = np.concatenate(all_features, axis=0)
  all_labels = np.concatenate(all_labels, axis=0)
  all_sex = np.concatenate(all_sex, axis=0)
  return all_features, all_labels, all_sex

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

adversarial_model = AdversarialDenseNet121().to(DEVICE)
adversarial_model.load_state_dict(torch.load("/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_sex_age_lambda1.pth"))
adversarial_model.eval()
print("Adversarial model loaded successfully")
print(f"Type: {type(adversarial_model)}")

X_train_features, y_train_disease, y_train_sex = extract_features(adversarial_model, train_loader, DEVICE)

X_test_features, y_test_disease, y_test_sex = extract_features(adversarial_model, test_loader, DEVICE)
# y_test_sex = test_df["tasks/patient sex"].values

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

pca = PCA(n_components=20)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)


probe = LogisticRegression(max_iter=2000)
probe.fit(X_train_pca, y_train_sex.ravel())

sex_probs = probe.predict_proba(X_test_pca)[:, 1]
auc = roc_auc_score(y_test_sex, sex_probs)

print("Sex Probe AUC:", round(auc, 4))

Adversarial model loaded successfully
Type: <class '__main__.AdversarialDenseNet121'>
Sex Probe AUC: 0.6042


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from scipy import stats
import numpy as np
def run_cv(X, y, n_splits=5, n_components=20):

  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  aucs = []

  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    probe = LogisticRegression(max_iter=2000)
    probe.fit(X_train_pca, y_train.ravel())
    auc_scores = roc_auc_score(y_val, probe.predict_proba(X_val_pca)[:, 1])
    aucs.append(auc_scores)
    print(f"  Fold {fold+1}: AUC = {auc_scores:.4f}")

  return np.array(aucs)

In [ ]:
def get_feature_importance(X_train, y_train, n_components=20):
    """
    Fit probe on FULL train set and extract weights.
    Run this SEPARATELY after CV.
    """
    # Scale
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # PCA
    pca            = PCA(n_components=n_components)
    X_train_pca    = pca.fit_transform(X_train_scaled)

    # Fit probe on full train set
    probe = LogisticRegression(max_iter=2000)
    probe.fit(X_train_pca, y_train.ravel())

    # Get weights
    weights_pca = probe.coef_[0]  # shape (20,)

    # Print weights per PC
    print("Weights per PCA component:")
    print(f"{'PC':<6} {'Weight':>10} {'Abs Weight':>12}")
    print("-" * 30)
    for i, w in enumerate(weights_pca):
        print(f"PC{i+1:<4} {w:>10.4f} {abs(w):>12.4f}")

    # Project back to 1024-d
    importance_1024 = weights_pca @ pca.components_

    # Top 10 most important features
    top10 = np.argsort(np.abs(importance_1024))[-10:][::-1]
    print("\nTop 10 most important features:")
    print(f"{'Feature':<10} {'Importance':>12}")
    print("-" * 25)
    for idx in top10:
        print(f"Feature {idx:<4} {importance_1024[idx]:>12.4f}")

    return weights_pca, importance_1024

In [ ]:
print("=" * 40)
print("BASELINE - Sex - 5-fold CV")
print("=" * 40)
aucs_base_sex = run_cv(X_train_features, y_train_sex)
print(f"Mean: {aucs_base_sex.mean():.4f}")
print(f"Std:  {aucs_base_sex.std():.4f}")

# Step 2: Get weights separately
print("\n" + "=" * 40)
print("BASELINE - Sex - Feature Importance")
print("=" * 40)
weights_base, importance_base = get_feature_importance(
    X_train_features, y_train_sex
)

BASELINE - Sex - 5-fold CV
  Fold 1: AUC = 0.6279
  Fold 2: AUC = 0.6283
  Fold 3: AUC = 0.6375
  Fold 4: AUC = 0.6247
  Fold 5: AUC = 0.6216
Mean: 0.6280
Std:  0.0053

BASELINE - Sex - Feature Importance
Weights per PCA component:
PC         Weight   Abs Weight
------------------------------
PC1       -0.0003       0.0003
PC2        0.0113       0.0113
PC3       -0.0277       0.0277
PC4       -0.0305       0.0305
PC5        0.0555       0.0555
PC6       -0.0060       0.0060
PC7       -0.0363       0.0363
PC8        0.0291       0.0291
PC9        0.2404       0.2404
PC10       0.0277       0.0277
PC11       0.0540       0.0540
PC12       0.0950       0.0950
PC13       0.0176       0.0176
PC14      -0.1064       0.1064
PC15      -0.0558       0.0558
PC16      -0.1259       0.1259
PC17      -0.1704       0.1704
PC18      -0.0784       0.0784
PC19       0.0509       0.0509
PC20      -0.2642       0.2642

Top 10 most important features:
Feature      Importance
-------------------------
Fea

In [ ]:
# ── SEX probe on combined model ──
print("\n" + "=" * 40)
print("DANN-BOTH - Sex - 5-fold CV")
print("=" * 40)
aucs_dann_both_sex = run_cv(X_train_features, y_train_sex)
print(f"Mean: {aucs_dann_both_sex.mean():.4f}")
print(f"Std:  {aucs_dann_both_sex.std():.4f}")

path = '/content/drive/MyDrive/IndividualProject/cv_results/dann_both'
np.save(f'{path}/lr_sex.npy', aucs_dann_both_sex)
print("Saved!")



DANN-BOTH - Sex - 5-fold CV
  Fold 1: AUC = 0.6238
  Fold 2: AUC = 0.6203
  Fold 3: AUC = 0.6350
  Fold 4: AUC = 0.6197
  Fold 5: AUC = 0.6368
Mean: 0.6271
Std:  0.0073


In [ ]:
from sklearn.svm import SVC
def run_cv_svm(X, y, n_splits=5, n_components=20):

  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  aucs_svm = []

  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    svm_rbf = SVC(kernel='rbf', probability=True)
    svm_rbf.fit(X_train_pca, y_train.ravel())
    auc_rbf = roc_auc_score(
        y_val.ravel(),
        svm_rbf.predict_proba(X_val_pca)[:, 1]
    )
    aucs_svm.append(auc_rbf)
    print(f"  Fold {fold+1}: SVM-RBF = {auc_rbf:.4f}")

  return np.array(aucs_svm)

In [ ]:
# DANN notebook - Sex
print("=" * 40)
print(" Adversarial - Sex - 5-fold CV")
print("=" * 40)
svm_adv_sex = run_cv_svm(X_train_features, y_train_sex)
for i, auc in enumerate(svm_adv_sex):
    print(f"  Fold {i+1}: AUC = {auc:.4f}")
print(f"SVM-RBF:  {svm_adv_sex.mean():.4f} ± {svm_adv_sex.std():.4f}")

path = '/content/drive/MyDrive/IndividualProject/cv_results/dann_both'
np.save(f'{path}/rbf_sex.npy', svm_adv_sex)
print("Saved!")

 Adversarial - Sex - 5-fold CV
  Fold 1: SVM-RBF = 0.6876
  Fold 2: SVM-RBF = 0.6890
  Fold 3: SVM-RBF = 0.6994
  Fold 4: SVM-RBF = 0.6853
  Fold 5: SVM-RBF = 0.6801
  Fold 1: AUC = 0.6876
  Fold 2: AUC = 0.6890
  Fold 3: AUC = 0.6994
  Fold 4: AUC = 0.6853
  Fold 5: AUC = 0.6801
SVM-RBF:  0.6883 ± 0.0063
Saved!


# Age Probing


In [ ]:

# define extract features functions
def extract_features_and_age(model, loader, device):

  model.eval()
  all_features = []
  all_labels = []
  all_age = []

  with torch.no_grad():
    for imgs, labels, _, age in loader:
      imgs = imgs.to(device)

      features = model.extract(imgs)

      all_features.append(features.cpu().numpy())
      all_labels.append(labels.numpy())
      all_age.append(age.numpy())


  all_features = np.concatenate(all_features, axis=0)
  all_labels = np.concatenate(all_labels, axis=0)
  all_age = np.concatenate(all_age, axis=0)
  return all_features, all_labels, all_age

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

adversarial_model = AdversarialDenseNet121().to(DEVICE)
adversarial_model.load_state_dict(torch.load("/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_sex_age_lambda1.pth"))
adversarial_model.eval()
print("Adversarial model loaded successfully")
print(f"Type: {type(adversarial_model)}")

X_train_age, y_train_disease, y_train_age = extract_features_and_age(adversarial_model, valid_loader, DEVICE)

X_test_age, y_test_disease, y_test_age = extract_features_and_age(adversarial_model, test_loader, DEVICE)


scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_age)
X_test_scaled = scaler.transform(X_test_age)

pca = PCA(n_components=20)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"Variance explained by 20 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%")

age_probe = LogisticRegression(max_iter=2000)
age_probe.fit(X_train_pca, y_train_age.ravel())

age_probs = age_probe.predict_proba(X_test_pca)[:, 1]
auc_age = roc_auc_score(y_test_age, age_probs)

print("Age Probe AUC:", round(auc_age, 4))

Adversarial model loaded successfully
Type: <class '__main__.AdversarialDenseNet121'>
Variance explained by 20 PCs: 95.6%
Age Probe AUC: 0.6131


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold
from scipy import stats
import numpy as np

def run_cv(X, y, n_splits=5, n_components=20):

  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  aucs = []

  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    probe = LogisticRegression(max_iter=2000)
    probe.fit(X_train_pca, y_train.ravel())
    auc_scores = roc_auc_score(y_val.ravel(), probe.predict_proba(X_val_pca)[:, 1])
    aucs.append(auc_scores)
    print(f"  Fold {fold+1}: AUC = {auc_scores:.4f}")

  return np.array(aucs)

In [ ]:
def get_feature_importance(X_train, y_train, n_components=20):
    """
    Fit probe on FULL train set and extract weights.
    Run this SEPARATELY after CV.
    """
    # Scale
    scaler         = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    # PCA
    pca            = PCA(n_components=n_components)
    X_train_pca    = pca.fit_transform(X_train_scaled)

    # Fit probe on full train set
    probe = LogisticRegression(max_iter=2000)
    probe.fit(X_train_pca, y_train.ravel())

    # Get weights
    weights_pca = probe.coef_[0]  # shape (20,)

    # Print weights per PC
    print("Weights per PCA component:")
    print(f"{'PC':<6} {'Weight':>10} {'Abs Weight':>12}")
    print("-" * 30)
    for i, w in enumerate(weights_pca):
        print(f"PC{i+1:<4} {w:>10.4f} {abs(w):>12.4f}")

    # Project back to 1024-d
    importance_1024 = weights_pca @ pca.components_

    # Top 10 most important features
    top10 = np.argsort(np.abs(importance_1024))[-10:][::-1]
    print("\nTop 10 most important features:")
    print(f"{'Feature':<10} {'Importance':>12}")
    print("-" * 25)
    for idx in top10:
        print(f"Feature {idx:<4} {importance_1024[idx]:>12.4f}")

    return weights_pca, importance_1024

In [ ]:
# ── AGE probe on combined model ──
print("=" * 40)
print("DANN-BOTH - Age - 5-fold CV")
print("=" * 40)
aucs_dann_both_age = run_cv(X_train_age, y_train_age)
print(f"Mean: {aucs_dann_both_age.mean():.4f}")
print(f"Std:  {aucs_dann_both_age.std():.4f}")

path = '/content/drive/MyDrive/IndividualProject/cv_results/dann_both'
np.save(f'{path}/lr_age.npy', aucs_dann_both_age)
print("Saved!")

DANN-BOTH - Age - 5-fold CV
  Fold 1: AUC = 0.6554
  Fold 2: AUC = 0.6768
  Fold 3: AUC = 0.6779
  Fold 4: AUC = 0.6685
  Fold 5: AUC = 0.6701
Mean: 0.6698
Std:  0.0080
Saved!


In [ ]:
# ── AGE probe on combined model ──
print("=" * 40)
print("DANN-BOTH - Age - 5-fold CV")
print("=" * 40)
aucs_dann_both_age = run_cv(X_train_age, y_train_age)
print(f"Mean: {aucs_dann_both_age.mean():.4f}")
print(f"Std:  {aucs_dann_both_age.std():.4f}")

DANN-BOTH - Age - 5-fold CV
  Fold 1: AUC = 0.6556
  Fold 2: AUC = 0.6767
  Fold 3: AUC = 0.6781
  Fold 4: AUC = 0.6690
  Fold 5: AUC = 0.6704
Mean: 0.6700
Std:  0.0080


In [ ]:
print("\n" + "=" * 40)
print("BASELINE - Sex - Feature Importance")
print("=" * 40)
weights_base, importance_base = get_feature_importance(
    X_train_age, y_train_age
)


BASELINE - Sex - Feature Importance
Weights per PCA component:
PC         Weight   Abs Weight
------------------------------
PC1       -0.0206       0.0206
PC2       -0.0033       0.0033
PC3       -0.0330       0.0330
PC4       -0.0507       0.0507
PC5       -0.0350       0.0350
PC6        0.0162       0.0162
PC7       -0.0364       0.0364
PC8       -0.0065       0.0065
PC9       -0.0030       0.0030
PC10      -0.0383       0.0383
PC11       0.0157       0.0157
PC12      -0.0597       0.0597
PC13      -0.1051       0.1051
PC14      -0.0543       0.0543
PC15      -0.0524       0.0524
PC16      -0.0257       0.0257
PC17      -0.2164       0.2164
PC18      -0.0856       0.0856
PC19       0.0907       0.0907
PC20       0.0521       0.0521

Top 10 most important features:
Feature      Importance
-------------------------
Feature 193       -0.0897
Feature 441       -0.0796
Feature 115       -0.0763
Feature 161        0.0686
Feature 224       -0.0524
Feature 269       -0.0497
Feature 92     

In [ ]:
from sklearn.svm import SVC
def run_cv_svm(X, y, n_splits=5, n_components=20):

  skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
  aucs_svm = []

  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):

    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)

    pca = PCA(n_components=n_components)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_val_pca = pca.transform(X_val_scaled)

    svm_rbf = SVC(kernel='rbf', probability=True)
    svm_rbf.fit(X_train_pca, y_train.ravel())
    auc_rbf = roc_auc_score(
        y_val.ravel(),
        svm_rbf.predict_proba(X_val_pca)[:, 1]
    )
    aucs_svm.append(auc_rbf)
    print(f"  Fold {fold+1}: SVM-RBF = {auc_rbf:.4f}")

  return np.array(aucs_svm)

In [ ]:
# ── AGE only ──
print("=" * 40)
print("DANN-AGE - Age - 5-fold CV")
print("=" * 40)
aucs_dann_age = run_cv_svm(X_train_age, y_train_age)
print(f"Mean: {aucs_dann_age.mean():.4f}")
print(f"Std:  {aucs_dann_age.std():.4f}")

path = '/content/drive/MyDrive/IndividualProject/cv_results/dann_both'
np.save(f'{path}/svm_age.npy', aucs_dann_age)
print("Saved!")

DANN-AGE - Age - 5-fold CV
  Fold 1: SVM-RBF = 0.6627
  Fold 2: SVM-RBF = 0.6904
  Fold 3: SVM-RBF = 0.6759
  Fold 4: SVM-RBF = 0.6694
  Fold 5: SVM-RBF = 0.6668
Mean: 0.6730
Std:  0.0097
Saved!


In [ ]:
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
import numpy as np
from tqdm import tqdm

# 1. Load the model and weights
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model = AdversarialDenseNet121().to(DEVICE)
model.load_state_dict(torch.load("/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_20260301_173719.pth")) # Update this path
model.eval()

# 2. Freeze the backbone
for param in model.parameters():
    param.requires_grad = False


# 3. Define the Linear Probe
# DenseNet-121's second-to-last layer output is 1024
age_probe = nn.Linear(1024, 1).to(DEVICE)
optimizer = torch.optim.Adam(age_probe.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

# 4. Train the Probe on the features
print("--- Training Linear Probe for Age Leakage ---")
for epoch in range(5): # 5-10 epochs is usually enough for a linear probe
    age_probe.train()
    epoch_loss = 0
    for images, _, _, age in valid_loader:
        images = images.to(DEVICE)
        age = age.to(DEVICE).float().view(-1, 1)

        # Extract frozen features
        with torch.no_grad():
            features = model.extract(images)

        optimizer.zero_grad()
        outputs = age_probe(features)
        loss = criterion(outputs, age)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {epoch_loss/len(train_loader):.4f}")

# 5. Evaluate the Leakage (AUC)
def evaluate_leakage(probe, feature_extractor, loader):
    probe.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, _, _, age in loader:
            features = feature_extractor.extract(images.to(DEVICE))
            preds = torch.sigmoid(probe(features))
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(age.numpy())

    return roc_auc_score(all_labels, all_preds)

final_leakage_auc = evaluate_leakage(age_probe, model, test_loader)

print("\n" + "="*40)
print(f"FINAL AGE LEAKAGE AUC: {final_leakage_auc:.4f}")
print("="*40)
if final_leakage_auc < 0.60:
    print("Interpretation: Success! Features are highly Age-Invariant.")
elif final_leakage_auc > 0.80:
    print("Interpretation: High Leakage. Age is still easily identifiable.")

# GradCam

In [ ]:
pip install grad-cam

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AdversarialDenseNet121().to(DEVICE)
checkpoint = torch.load( "/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_20260301_173719.pth", map_location=DEVICE)
model.load_state_dict(checkpoint)
model.to(DEVICE)
model.eval()

In [ ]:
def collect_confident_tp_age(
    loader,
    model,
    device,
    target_age_group,   # 0 = <50 , 1 = ≥50
    n_cases=5,
    threshold=0.47817206,
    is_adversarial=False
):

    model.eval()
    collected = []

    with torch.no_grad():

        for imgs, labels, _, age in loader:

            imgs = imgs.to(device)

            labels = labels.squeeze().to(device)
            age = age.squeeze().to(device)

            if is_adversarial:
                logits, _ = model(imgs, lambd=0.0)
            else:
                logits = model(imgs)

            probs = torch.sigmoid(logits).squeeze()

            mask = (
                (labels == 1) &
                (probs > threshold) &
                (age == target_age_group)
            )

            if mask.sum() > 0:

                for img, prob in zip(imgs[mask], probs[mask]):

                    collected.append((img.unsqueeze(0).cpu(), prob.item()))

                    if len(collected) >= n_cases:
                        return [x[0] for x in collected]

    return [x[0] for x in collected]

In [ ]:
young_cases = collect_confident_tp_age(
    test_loader,
    model,
    DEVICE,
    target_age_group=0,
    is_adversarial=True
)

old_cases = collect_confident_tp_age(
    test_loader,
    model,
    DEVICE,
    target_age_group=1,
    is_adversarial=True
)

In [ ]:
target_layers = [model.features.denseblock4]

In [ ]:
class TaskWrapper(torch.nn.Module):

    def __init__(self, adv_model):
        super().__init__()
        self.model = adv_model

    def forward(self, x):

        task_logit, _ = self.model(x, lambd=0.0)   # ⭐ KEY FIX

        return task_logit

In [ ]:
wrapped_model = TaskWrapper(model)

cam = GradCAM(
    model=wrapped_model,
    target_layers=target_layers
)

In [ ]:
targets = [ClassifierOutputTarget(0)]

def run_cam_list(case_list):

    vis_list = []

    for img_tensor in case_list:

        cam_map = cam(
            input_tensor=img_tensor,
            targets=targets
        )[0]

        img = denorm(img_tensor)

        vis = show_cam_on_image(
            img,
            cam_map,
            use_rgb=True
        )

        vis_list.append(vis)

    return vis_list

In [ ]:
def denorm(img_tensor):

    img = img_tensor[0].detach().cpu().numpy()
    img = np.transpose(img, (1,2,0))

    mean = np.array([0.485,0.456,0.406])
    std = np.array([0.229,0.224,0.225])

    img = std * img + mean
    img = np.clip(img,0,1)

    return img

In [ ]:
male_vis = run_cam_list(young_cases)
female_vis = run_cam_list(old_cases)

In [ ]:
plt.figure(figsize=(12,6))

for i in range(5):

    plt.subplot(2,5,i+1)
    plt.imshow(male_vis[i])
    plt.title("Young")
    plt.axis("off")

    plt.subplot(2,5,i+6)
    plt.imshow(female_vis[i])
    plt.title("Old")
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
!pip install grad-cam
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image

In [ ]:
class TaskWrapper(nn.Module):
    """Wraps adversarial model so GradCAM only sees the task head output"""
    def __init__(self, model):
        super(TaskWrapper, self).__init__()
        self.model = model

    def forward(self, x):
        # lambd=0 means GRL has no effect during inference
        task_output, _ = self.model(x, lambd=0)
        return task_output

In [ ]:
BEST_ADV_MODEL_PATH = "/content/drive/MyDrive/IndividualProject/best_model_binary_adversarial_atelectasis_20260301_173719.pth"

adv_model = AdversarialDenseNet121().to(DEVICE)
adv_model.load_state_dict(torch.load(BEST_ADV_MODEL_PATH, map_location=DEVICE))
adv_model.eval()

# wrap it for GradCAM
wrapped_adv_model = TaskWrapper(adv_model).to(DEVICE)
wrapped_adv_model.eval()

In [ ]:
# DenseNet121 target layer — last dense block before classifier
target_layer_adv = [wrapped_adv_model.model.features.denseblock4]
cam_adv = GradCAM(model=wrapped_adv_model, target_layers=target_layer_adv)

def denorm(tensor):
    """Convert normalised tensor back to 0-1 range for visualisation"""
    img = tensor.clone()
    img = img - img.min()
    img = img / img.max()
    return img.permute(1, 2, 0).numpy()  # (H, W, 3)

In [ ]:
young_tp_adv = []
old_tp_adv = []
MAX_PER_GROUP = 5
THRESHOLD = 0.5

with torch.no_grad():
    for images, labels, sex, age in test_loader:  # ← unpack age too
        images = images.to(DEVICE)
        task_output, _ = adv_model(images, lambd=0)
        outputs = torch.sigmoid(task_output)
        preds = (outputs > THRESHOLD).float().cpu()

        for i in range(len(images)):
            label = labels[i].item()
            pred = preds[i].item()
            a = age[i].item()  # 0 = <50, 1 = >=50

            if label == 1 and pred == 1:
                img_cpu = images[i].cpu()
                if a == 0 and len(young_tp_adv) < MAX_PER_GROUP:
                    young_tp_adv.append(img_cpu)
                elif a == 1 and len(old_tp_adv) < MAX_PER_GROUP:
                    old_tp_adv.append(img_cpu)

        if len(young_tp_adv) >= MAX_PER_GROUP and len(old_tp_adv) >= MAX_PER_GROUP:
            break

print(f"Collected {len(young_tp_adv)} young TPs, {len(old_tp_adv)} old TPs")

In [ ]:
import matplotlib.pyplot as plt
def plot_gradcam_grid(cases, title, cam, device):
    fig, axes = plt.subplots(1, len(cases), figsize=(4 * len(cases), 4))
    if len(cases) == 1:
        axes = [axes]

    for ax, img_tensor in zip(axes, cases):
        input_tensor = img_tensor.unsqueeze(0).to(device)

        # generate CAM — targets=None uses the highest scoring class
        grayscale_cam = cam(input_tensor=input_tensor, targets=None)
        grayscale_cam = grayscale_cam[0]  # (H, W)

        rgb_img = denorm(img_tensor)  # (H, W, 3)

        # overlay heatmap on image
        visualization = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)

        ax.imshow(visualization)
        ax.axis('off')

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(f"{title.replace(' ', '_')}.png", dpi=150)
    plt.show()

In [ ]:
plot_gradcam_grid(young_tp_adv, "DANN — Age <50 True Positives",  cam_adv, DEVICE)
plot_gradcam_grid(old_tp_adv,   "DANN — Age >=50 True Positives", cam_adv, DEVICE)